# Portfolio Builder

Interactive notebook for exploring the efficient frontier and the Capital Allocation Line (CAL).

1. Set `INPUT_FILE` and `SAMPLE_SIZE` in the **Configuration** cell.
2. Run all cells top-to-bottom (`Run All`).

In [1]:
import sys, os

# Notebook lives in src/; project root is one level up
NOTEBOOK_DIR = os.path.abspath('.')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [2]:
import src.PortfolioBuilderObjects as obj
import src.graph_utils.Graphs as gr
from src.PortfolioBuilder import read_excel_input, create_portfolios, compute_sharpe

## Configuration

In [3]:
INPUT_FILE  = os.path.join(PROJECT_ROOT, 'input', 'input.xlsx')
SAMPLE_SIZE = 50_000               # reduce for speed, increase for accuracy

## 1. Load Data

In [4]:
asset_classes, corr_matrix, rf, available_rate, period, user_portfolios = read_excel_input(INPUT_FILE, 'excel')

print(f'Loaded {len(asset_classes)} asset classes')
print(f'Risk-free rate : {rf}%  |  Period : {period}\n')
for ac in asset_classes:
    print(ac)

Loaded 15 asset classes
Risk-free rate : 5.04%  |  Period : Weekly

AssetClass = [VOO, E(r) = 0.28%, sd = 0.33%]
AssetClass = [VXUS, E(r) = 0.15%, sd = 0.31%]
AssetClass = [SCHK, E(r) = 0.20%, sd = 0.30%]
AssetClass = [VRE.TO, E(r) = 0.07%, sd = 0.20%]
AssetClass = [VNQ, E(r) = 0.08%, sd = 0.22%]
AssetClass = [VTV, E(r) = 0.15%, sd = 0.22%]
AssetClass = [SCHD, E(r) = 0.32%, sd = 0.29%]
AssetClass = [VCN.TO, E(r) = 0.18%, sd = 0.31%]
AssetClass = [VTI, E(r) = 0.33%, sd = 0.41%]
AssetClass = [VAB.TO, E(r) = 0.17%, sd = 0.30%]
AssetClass = [VB, E(r) = 0.14%, sd = 0.23%]
AssetClass = [GLD, E(r) = 0.01%, sd = 0.20%]
AssetClass = [QQQ, E(r) = 0.34%, sd = 0.37%]
AssetClass = [VPU, E(r) = 0.20%, sd = 0.29%]
AssetClass = [VCE.TO, E(r) = 0.05%, sd = 0.34%]


## 2. Generate Random Portfolios

In [5]:
print(f'Generating {SAMPLE_SIZE:,} random portfolios...')
portfolios    = create_portfolios(asset_classes, corr_matrix, SAMPLE_SIZE)
portfolios_df = obj.convert_portfolios_into_df(portfolios)
print('Done.')
portfolios_df.head()

Generating 50,000 random portfolios...
Done.


,Name,E(r),sd,Portfolio Object
0,Portfolio1,0.199861,0.221269,"Portfolio = [Portfolio1, composition = {\n ..."
1,Portfolio2,0.185705,0.200324,"Portfolio = [Portfolio2, composition = {\n ..."
2,Portfolio3,0.195765,0.226215,"Portfolio = [Portfolio3, composition = {\n ..."
3,Portfolio4,0.188043,0.208576,"Portfolio = [Portfolio4, composition = {\n ..."
4,Portfolio5,0.158220,0.197059,"Portfolio = [Portfolio5, composition = {\n ..."


## 3. Sharpe Ratios

In [6]:
portfolios_df = compute_sharpe(portfolios_df, rf)
portfolios_df.sort_values('Sharpe', ascending=False)[['Name', 'E(r)', 'sd', 'Sharpe']].head(10)

,Name,E(r),sd,Sharpe
0,Portfolio1,0.199861,0.221269,0.0
1,Portfolio2,0.185705,0.200324,0.0
2,Portfolio3,0.195765,0.226215,0.0
3,Portfolio4,0.188043,0.208576,0.0
4,Portfolio5,0.158220,0.197059,0.0
5,Portfolio6,0.164515,0.195276,0.0
6,Portfolio7,0.192872,0.212958,0.0
7,Portfolio8,0.176077,0.205176,0.0
8,Portfolio9,0.199439,0.210145,0.0
9,Portfolio10,0.193600,0.220504,0.0


## 4. Efficient Frontier & Capital Allocation Line

- **Grey dots** — randomly generated portfolios forming the efficient frontier
- **Green dot** — optimal portfolio (maximum Sharpe ratio)
- **Blue line** — Capital Allocation Line (CAL)
- **Light-blue dot** — risk-free asset
- **Coloured dots** — your custom portfolios from the input file

In [7]:
fig = gr.get_scatter_plot(
    portfolios_df,
    title=f'Efficient Frontier — {SAMPLE_SIZE:,} portfolios ({period.lower()} returns)'
)
fig, optimal_portfolio = gr.add_CAL(fig, portfolios_df, rf)
fig = gr.add_user_portfolios(fig, user_portfolios)
fig.show()

## 5. Optimal Portfolio

In [8]:
print('\n~~~ Optimal Portfolio (maximum Sharpe ratio) ~~~\n')
print(optimal_portfolio)


~~~ Optimal Portfolio (maximum Sharpe ratio) ~~~

Portfolio = [Portfolio1, composition = {
      5.09% : AssetClass = [VOO, E(r) = 0.28%, sd = 0.33%],
     12.80% : AssetClass = [VXUS, E(r) = 0.15%, sd = 0.31%],
      6.22% : AssetClass = [SCHK, E(r) = 0.20%, sd = 0.30%],
      4.57% : AssetClass = [VRE.TO, E(r) = 0.07%, sd = 0.20%],
      4.86% : AssetClass = [VNQ, E(r) = 0.08%, sd = 0.22%],
      2.53% : AssetClass = [VTV, E(r) = 0.15%, sd = 0.22%],
      8.87% : AssetClass = [SCHD, E(r) = 0.32%, sd = 0.29%],
      4.90% : AssetClass = [VCN.TO, E(r) = 0.18%, sd = 0.31%],
     12.54% : AssetClass = [VTI, E(r) = 0.33%, sd = 0.41%],
      6.42% : AssetClass = [VAB.TO, E(r) = 0.17%, sd = 0.30%],
      9.25% : AssetClass = [VB, E(r) = 0.14%, sd = 0.23%],
      2.09% : AssetClass = [GLD, E(r) = 0.01%, sd = 0.20%],
      5.92% : AssetClass = [QQQ, E(r) = 0.34%, sd = 0.37%],
      9.48% : AssetClass = [VPU, E(r) = 0.20%, sd = 0.29%],
      4.46% : AssetClass = [VCE.TO, E(r) = 0.05%, sd = 0.

## 6. Save Chart (optional)

Uncomment to export the figure as a self-contained HTML file.

In [9]:
# gr.save_fig_as_html(fig, 'efficient_frontier.html')